In [104]:
import requests
import polars as pl


In [105]:
data = requests.get("https://ogd-static.voteinfo-app.ch/v4/ogd/kommunale_resultate_2026_03_08.json").json()

In [106]:
results = list(filter(lambda row: row['vorlagenId']==369491, data['kantone'][1]['vorlagen']))[0]

In [107]:
mapping_str = """acf3a0b6-1d74-44df-aeec-188f3681f168/Simone_Brander
36f72707-3f2f-46bb-aa40-83ab6173b610/Raphael_Golta
aa34ce4a-3d5b-4a1a-a4de-b0ee9642199f/Céline_Widmer
785ff6ff-ec72-4792-8910-efd67bf2ed15/Tobias_Langenegger
4d4d3a9e-8d20-4124-9021-9382057bd4af/Daniel_Leupi
4fbd8a2a-ee02-4946-bf09-73cefad6258b/Karin_Rykart
33b610f1-ef4d-4f9f-9c57-a66dc70bffc5/Balthasar_Glättli
29b4f78a-57be-4a3d-b5b4-4d1f31e13496/Andreas_Hauri
ca92fe7d-0487-4749-996d-ea59ce764184/Serap_Kahriman
be1f7c49-dbdd-4776-a90e-69fd7f62203a/Michael_Baumer
b4d85396-7d12-4171-9dc9-c0e971f22260/Përparim_Avdili
0ea5d814-8830-4f71-a88d-4b89eea23423/Marita_Verbali
deec8ffb-f7db-45e2-85a4-22bf919eb3ff/Ueli_Bamert
4b9800b6-c007-4f9d-993a-ebc0681891dd/Tanja_Maag
8793c0b0-fd84-46fb-a34d-d02973f6817c/Karin_Weyermann
d4dbce62-e469-45a9-b598-e3c44fe8143f/Sandra_Gallizzi"""

mapping = dict(map(lambda line: line.split("/"), mapping_str.split("\n")))


In [108]:
mapping

{'acf3a0b6-1d74-44df-aeec-188f3681f168': 'Simone_Brander',
 '36f72707-3f2f-46bb-aa40-83ab6173b610': 'Raphael_Golta',
 'aa34ce4a-3d5b-4a1a-a4de-b0ee9642199f': 'Céline_Widmer',
 '785ff6ff-ec72-4792-8910-efd67bf2ed15': 'Tobias_Langenegger',
 '4d4d3a9e-8d20-4124-9021-9382057bd4af': 'Daniel_Leupi',
 '4fbd8a2a-ee02-4946-bf09-73cefad6258b': 'Karin_Rykart',
 '33b610f1-ef4d-4f9f-9c57-a66dc70bffc5': 'Balthasar_Glättli',
 '29b4f78a-57be-4a3d-b5b4-4d1f31e13496': 'Andreas_Hauri',
 'ca92fe7d-0487-4749-996d-ea59ce764184': 'Serap_Kahriman',
 'be1f7c49-dbdd-4776-a90e-69fd7f62203a': 'Michael_Baumer',
 'b4d85396-7d12-4171-9dc9-c0e971f22260': 'Përparim_Avdili',
 '0ea5d814-8830-4f71-a88d-4b89eea23423': 'Marita_Verbali',
 'deec8ffb-f7db-45e2-85a4-22bf919eb3ff': 'Ueli_Bamert',
 '4b9800b6-c007-4f9d-993a-ebc0681891dd': 'Tanja_Maag',
 '8793c0b0-fd84-46fb-a34d-d02973f6817c': 'Karin_Weyermann',
 'd4dbce62-e469-45a9-b598-e3c44fe8143f': 'Sandra_Gallizzi'}

In [109]:

rows = []
for zaehlkreis in results['zaehlkreise']:
    for kandidat in zaehlkreis['resultat']['kandidaten']:
        rows.append({
            "zaehlkreis": zaehlkreis['geoLevelname'],
            "kandidat": mapping.get(kandidat['kandidatNummer'], "Andere"),
            "stimmen": kandidat['stimmen']
        })

df = pl.DataFrame(rows)

In [110]:
order = """Raphael Golta
Përparim Avdili
Ueli Bamert
Serap Kahriman
Andere Kandidierende
""".split('\n')

df_order = pl.DataFrame({'kandidat': mapping.values()})

In [111]:
df_order

kandidat
str
"""Simone_Brander"""
"""Raphael_Golta"""
"""Céline_Widmer"""
"""Tobias_Langenegger"""
"""Daniel_Leupi"""
…
"""Marita_Verbali"""
"""Ueli_Bamert"""
"""Tanja_Maag"""


In [112]:
df_wide = df.pivot(index="kandidat", on="zaehlkreis", values="stimmen", aggregate_function="sum")

In [113]:
df_wide

kandidat,Zürich Kreise 1 und 2,Zürich Kreis 3,Zürich Kreis 4 und 5,Zürich Kreis 6,Zürich Kreise 7 und 8,Zürich Kreis 9,Zürich Kreis 10,Zürich Kreis 11,Zürich Kreis 12
str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Marita_Verbali""",0,3814,0,0,0,0,3858,4655,1450
"""Andere""",0,10136,0,0,0,0,9715,12240,4321
"""Andreas_Hauri""",0,5946,0,0,0,0,6297,6643,1980
"""Balthasar_Glättli""",0,7589,0,0,0,0,6866,6167,1992
"""Raphael_Golta""",0,9474,0,0,0,0,8671,8573,2864
…,…,…,…,…,…,…,…,…,…
"""Përparim_Avdili""",0,5070,0,0,0,0,5160,6239,2082
"""Michael_Baumer""",0,5556,0,0,0,0,5968,7289,2281
"""Serap_Kahriman""",0,4556,0,0,0,0,4461,4717,1324


In [114]:
df_final = df_order.join(df_wide, on='kandidat', how='left')
df_final

kandidat,Zürich Kreise 1 und 2,Zürich Kreis 3,Zürich Kreis 4 und 5,Zürich Kreis 6,Zürich Kreise 7 und 8,Zürich Kreis 9,Zürich Kreis 10,Zürich Kreis 11,Zürich Kreis 12
str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""Simone_Brander""",0,8551,0,0,0,0,7520,7543,2459
"""Raphael_Golta""",0,9474,0,0,0,0,8671,8573,2864
"""Céline_Widmer""",0,8612,0,0,0,0,7721,7255,2449
"""Tobias_Langenegger""",0,7800,0,0,0,0,6802,6461,2217
"""Daniel_Leupi""",0,8533,0,0,0,0,7948,7736,2453
…,…,…,…,…,…,…,…,…,…
"""Marita_Verbali""",0,3814,0,0,0,0,3858,4655,1450
"""Ueli_Bamert""",0,3247,0,0,0,0,3668,5319,1935
"""Tanja_Maag""",0,5739,0,0,0,0,4666,3884,1302


In [115]:
df_final.write_excel("results_sr.xlsx")